# 03 Modeling: tune & track

Goal: find the best model **reliably** and **never lose a result**.

The input is the cleaned data from `01_eda.ipynb` (`data/data_processed.csv`).

1. Score models with 5-fold **cross-validation** (macro F1)
2. **Tune** them with Hyperopt
3. **Log** every tuned model to MLflow
4. Load the winner back from MLflow and predict

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import numpy as np
import pandas as pd
import mlflow
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval
from hyperopt.pyll.base import scope

from personality.config import PROCESSED_DATA_PATH, RANDOM_STATE, CV_FOLDS, SCORING
from personality.data import split_features_target
from personality.pipeline import build_model_pipeline
from personality.tracking import setup_mlflow, log_model, get_best_run_id, load_model

pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
df = pd.read_csv(PROCESSED_DATA_PATH)
X, y = split_features_target(df)
print("Data:", X.shape)

Data: (19715, 22)


## 1. Cross-validation

One train/test split can be lucky or unlucky. Cross-validation splits the data into 5 parts,
trains 5 times (each part is the test set once) and averages the scores. That gives a steadier number.

`StratifiedKFold` keeps the class shares equal in every fold, like `stratify=y` in the split.

In [3]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
}

def cv_score(classifier) -> float:
    """Mean macro F1 of preprocessor + classifier over the CV folds."""
    pipe = build_model_pipeline(classifier)
    return cross_val_score(pipe, X, y, cv=cv, scoring=SCORING).mean()

baseline_scores = {"Baseline (always Moderate)": cv_score(DummyClassifier(strategy="most_frequent"))}
for name, classifier in models.items():
    baseline_scores[name] = cv_score(classifier)

pd.Series(baseline_scores, name="macro F1 (default settings)")

Baseline (always Moderate)   0.1500
Logistic Regression          0.7680
Random Forest                0.7486
Name: macro F1 (default settings), dtype: float64

## 2. Tuning with Hyperopt

Every model has settings (**hyperparameters**), e.g. how many trees a Random Forest has.
Hyperopt tries `MAX_EVALS` combinations from a **search space** and learns which areas score best.

- `objective(params)`: builds the model with these settings and returns the CV score.
  Hyperopt *minimizes*, so we return the **negative** macro F1 as `loss`.
- `hp.quniform(name, low, high, step)`: whole numbers in a range
- `hp.loguniform`: values spread on a log scale (0.01, 0.1, 1, 10 are equally likely)

In [4]:
search_spaces = {
    "Logistic Regression": {
        "C": hp.loguniform("C", np.log(0.01), np.log(10)),
    },
    "Random Forest": {
        "n_estimators": scope.int(hp.quniform("n_estimators", 100, 400, 50)),
        "max_depth": scope.int(hp.quniform("max_depth", 10, 40, 1)),
        "min_samples_leaf": scope.int(hp.quniform("min_samples_leaf", 1, 4, 1)),
    },
}

MAX_EVALS = 10  # small to keep runtime short; raise for a deeper search

## 3. Tune and log every model to MLflow

For each model: tune it, fit the best version on all data, and log
**parameters, CV score and the full fitted pipeline** to MLflow.
Nothing is lost when the kernel restarts.

In [5]:
setup_mlflow()

tuned_scores = {}
for name, classifier in models.items():
    print(f"Tuning {name} ...")

    def objective(params):
        score = cv_score(clone(classifier).set_params(**params))
        return {"loss": -score, "status": STATUS_OK}

    trials = Trials()
    best = fmin(objective, search_spaces[name], algo=tpe.suggest, max_evals=MAX_EVALS,
                trials=trials, rstate=np.random.default_rng(RANDOM_STATE))
    best_params = space_eval(search_spaces[name], best)
    best_score = -trials.best_trial["result"]["loss"]
    tuned_scores[name] = best_score

    final_model = build_model_pipeline(clone(classifier).set_params(**best_params))
    final_model.fit(X, y)

    with mlflow.start_run(run_name=name):
        mlflow.log_param("model", name)
        mlflow.log_params(best_params)
        mlflow.log_metric("f1_macro", best_score)
        log_model(final_model)

    print(f"  macro F1: {baseline_scores[name]:.4f} (default) -> {best_score:.4f} (tuned)  {best_params}")

2026/11/28 15:26:37 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/28 15:26:37 INFO mlflow.store.db.utils: Updating database tables


Tuning Logistic Regression ...
100%|██████████| 10/10 [00:14<00:00,  1.49s/trial, best loss: -0.7686110612285346]
  macro F1: 0.7680 (default) -> 0.7686 (tuned)  {'C': 2.760363161733829}
Tuning Random Forest ...
100%|██████████| 10/10 [00:40<00:00,  4.05s/trial, best loss: -0.7527796700244302]
  macro F1: 0.7486 (default) -> 0.7528 (tuned)  {'max_depth': 23, 'min_samples_leaf': 1, 'n_estimators': 350}


In [6]:
pd.DataFrame({
    "default": pd.Series(baseline_scores),
    "tuned": pd.Series(tuned_scores),
}).sort_values("tuned", ascending=False)

,default,tuned
Logistic Regression,0.7680,0.7686
Random Forest,0.7486,0.7528
Baseline (always Moderate),0.1500,NaN


## 4. Compare runs in the MLflow UI

Open a terminal **in the repo root** (where `mlflow.db` is) and run:

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Then open http://localhost:5000, select the experiment **personality-predictor** and sort by `f1_macro`.

## 5. Load the winner from MLflow and predict

`get_best_run_id()` finds the run with the highest macro F1, so no run id has to be copied by hand.
The app uses the same function. The loaded model is the **full pipeline**: raw answers in, personality type out.

In [7]:
best_run_id = get_best_run_id()
model = load_model(best_run_id)
print("Best run:", best_run_id)

example = pd.DataFrame([{
    "N1": 1, "N2": 5, "N3": 2, "N4": 5, "N5": 1, "N6": 1, "N7": 1, "N8": 1, "N9": 1, "N10": 1,
    "E1": 4, "E3": 5, "E4": 2, "E5": 5, "E7": 4, "E9": 5, "E10": 1,
    "C4": 1, "A4": 5,
    "age": 30, "gender": "Female", "hand": "Right",
}])
model.predict(example)[0]

Best run: c88b3267416f4d48881aed370b8442ef


'Resilient'

## Summary

| Model | Macro F1 default | Macro F1 tuned |
|---|---|---|
| Baseline (always Moderate) | 0.150 | |
| **Logistic Regression** | 0.768 | **0.768** |
| Random Forest | 0.749 | 0.753 |

- Cross-validated macro F1 (5 folds) is the single number used to compare models.
- **Logistic Regression wins**, even though it is the simpler model. The target is built from linear trait scores (sums of answers), which a linear model can represent directly.
- **Tuning barely helps** (+0.000 / +0.004). The limit is the information in the 19 questions (only one C and one A question), not the model settings.
- Every tuned model is logged in MLflow; the app loads the best run automatically via `get_best_run_id()`.